# Fotos → pixel art com Qwen/Qwen-Image
Usa os pesos solicitados com `QwenImageImg2ImgPipeline`; não é Qwen Edit nem um checkpoint fine-tunado.

**GPU NVIDIA/CUDA necessária.** Uma GPU pequena do Colab gratuito pode não comportar o modelo. Configure um runtime com GPU e memória suficientes. Os pesos são grandes e serão baixados na primeira execução.

No Colab, envie `aquario-qwen.zip` na primeira célula. Em Jupyter de uma VM, copie o ZIP para a pasta do notebook antes de executar. Este notebook não cria nem cobra uma VM automaticamente.

In [ ]:
from pathlib import Path
import zipfile, os
archive = Path("aquario-qwen.zip")
if not archive.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Copie aquario-qwen.zip para a pasta deste notebook.")
    uploaded = files.upload()
    if archive.name not in uploaded:
        raise FileNotFoundError("Envie o arquivo aquario-qwen.zip.")
work = Path("aquario-qwen").resolve()
work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (work / member.filename).resolve()
        if not target.is_relative_to(work):
            raise ValueError("Caminho inválido no ZIP")
    z.extractall(work)
os.chdir(work)
print("Pasta:", work)


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-qwen.txt"], check=True)


Se o notebook já havia importado Torch/Diffusers antes de instalar dependências, reinicie o kernel e volte à pasta extraída. Os subprocessos abaixo usam as bibliotecas recém-instaladas.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-c", "import torch; print('CUDA:',torch.cuda.is_available()); assert torch.cuda.is_available(), 'Selecione um runtime NVIDIA/CUDA'; print(torch.cuda.get_device_name(0)); print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/2**30,1))"], check=True)


In [ ]:
# Não baixa pesos nem gera imagens: confere as três entradas.
subprocess.run([sys.executable, "scripts/qwen_pixelart.py", "--dataset", "kauar_peixes.json", "--smoke", "--dry-run"], check=True)


## Gerar as três espécies
A primeira execução baixa Qwen e o segmentador opcional. Comece por este teste e avalie a identidade antes do lote. Nenhum resultado do Qwen foi pré-calculado neste pacote.

In [ ]:
STRENGTH = 0.65
STEPS = 40
RESOLUTION = 768
OFFLOAD = "model"  # "sequential" economiza VRAM, mas pode ser muito mais lento.
REMOVE_BACKGROUND = True
OUTPUT = "output/teste-qwen"
command = [sys.executable, "scripts/qwen_pixelart.py", "--dataset", "kauar_peixes.json", "--smoke",
           "--strength", str(STRENGTH), "--steps", str(STEPS), "--resolution", str(RESOLUTION),
           "--offload", OFFLOAD, "--output", OUTPUT]
if REMOVE_BACKGROUND:
    command += ["--remove-background"]
subprocess.run(command, check=True)


In [ ]:
from IPython.display import display
from PIL import Image
import json
report = json.loads(Path(OUTPUT, "report.json").read_text())
print(report)
for sid in report["success"] + report["cached"]:
    print(sid)
    display(Image.open(Path(OUTPUT, sid + ".input.png")).resize((384,384)))
    display(Image.open(Path(OUTPUT, sid + ".preview.png")))


## Sua própria foto (opcional)
Envie uma foto no Colab ou copie para esta VM. Execute a próxima célula somente se quiser transformar outra foto. Descreva a espécie em `SUBJECT`.

In [ ]:
PHOTO = ""  # Exemplo: "/content/meu-peixe.jpg"; vazio não executa.
SUBJECT = "small yellow aquarium fish with black vertical stripes"
if PHOTO:
    command = [sys.executable, "scripts/qwen_pixelart.py", "--input", PHOTO, "--subject", SUBJECT,
               "--strength", str(STRENGTH), "--offload", OFFLOAD, "--output", "output/minha-foto"]
    if REMOVE_BACKGROUND:
        command += ["--remove-background"]
    subprocess.run(command, check=True)


## Lote completo (opcional)
O smoke não dispara o catálogo inteiro. Só habilite depois de conferir os sprites e a capacidade da máquina.

In [ ]:
RUN_FULL_BATCH = False
if RUN_FULL_BATCH:
    command = [sys.executable, "scripts/qwen_pixelart.py", "--dataset", "kauar_peixes.json", "--all",
               "--strength", str(STRENGTH), "--steps", str(STEPS), "--resolution", str(RESOLUTION),
               "--offload", OFFLOAD, "--output", "output/catalogo-qwen"]
    if REMOVE_BACKGROUND:
        command += ["--remove-background"]
    subprocess.run(command, check=True)


In [ ]:
import shutil
result = shutil.make_archive("resultados-qwen", "zip", "output")
print("Resultados:", result)
try:
    from google.colab import files
    files.download(result)
except ImportError:
    pass  # Em Jupyter, baixe o ZIP pelo explorador de arquivos.
